In [9]:
import ast
import pandas as pd
import numpy as np

In [10]:
# Convert a cell into python list
def parse_list_cell(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None
        return ast.literal_eval(value)
    return value

# Return avg, std, min and max for a numeric pandas Series
def stats_for_series(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return {
        "average": s.mean() if len(s) else np.nan,
        "standard_deviation": s.std(ddof=1) if len(s) > 1 else 0.0 if len(s) == 1 else np.nan,
        "min": s.min() if len(s) else np.nan,
        "max": s.max() if len(s) else np.nan,
    }


# Compute avg, std, min and max for a pandas series where each cell contains a list. First each cell is converted to a Python list
# then the list is converted to a 2D Numpy array, verifies that all the lists have same lenght and computes statistics values accross rows.
def stats_for_list_series(series):
    parsed = series.dropna().map(parse_list_cell).tolist()
    if not parsed:
        return {
            "average": np.nan,
            "standard_deviation": np.nan,
            "min": np.nan,
            "max": np.nan,
        }

    arr = np.array(parsed, dtype = float)

    if arr.ndim != 2:
        raise ValueError("List column contains inconsistent list shapes.")

    return {
        "average": arr.mean(axis = 0).tolist(),
        "standard_deviation": (
            arr.std(axis = 0, ddof = 1).tolist() if len(arr) > 1 else [0.0] * arr.shape[1]
        ),
        "min": arr.min(axis = 0).tolist(),
        "max": arr.max(axis = 0).tolist(),
    }

# Collect rows in groups of 3
def summarize_csv_in_groups_of_three(
    df,
    output_csv,
    numeric_columns,
    list_columns,
    keep_columns,
    filter_column="test_number",
    filter_threshold=100,
    sort_by=None,
):

    df = df[df[filter_column] < filter_threshold].copy()

    if sort_by:
        df = df.sort_values(sort_by).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    df["counter"] = (df.index // 3) + 1

    results = []

    for counter, group in df.groupby("counter", sort=True):
        row = {"counter": counter}

        # Keep columns
        for col in keep_columns:
            row[col] = group.iloc[0][col]

        # Numeric columns
        for col in numeric_columns:
            stats = stats_for_series(group[col])
            row[f"{col}_average"] = stats["average"]
            row[f"{col}_standard_deviation"] = stats["standard_deviation"]
            row[f"{col}_min"] = stats["min"]
            row[f"{col}_max"] = stats["max"]

        # List columns
        for col in list_columns:
            stats = stats_for_list_series(group[col])
            row[f"{col}_average"] = stats["average"]
            row[f"{col}_standard_deviation"] = stats["standard_deviation"]
            row[f"{col}_min"] = stats["min"]
            row[f"{col}_max"] = stats["max"]

        # rows in this group
        row["rows_in_group"] = len(group)

        results.append(row)

    out_df = pd.DataFrame(results)
    
    if output_csv:
        csv_file = output_csv
        excel_file = str(output_csv).replace(".csv", ".xlsx")
    
        out_df.to_csv(csv_file, index=False, float_format="%.6f")
        out_df.to_excel(excel_file, index=False)

    return out_df

In [11]:
numeric_cols = [
    "execution_time_sec",
    "throughput_total_rows_sec",
    "throughput_rows_used_ml_sec",
    "execution_time_sec_train",
    "execution_time_sec_pred",
    "avg_rss_mib",
    "peak_rss_mib",
    "memory_percent_of_total",
    "avg_process_cpu_percent",
    "peak_process_cpu_percent",
    "logical_read_mib",
    "logical_write_mib",
    "physical_read_mib",
    "physical_write_mib",
]

list_cols = [
    "peak_system_cpu_per_core_percent",
    "avg_system_cpu_per_core_percent",
]

keep_cols = [
    "stage",
    "operation",
    "slice_name",
    "parquet_path",
    "total_files",
    "total_rows_in_dataset",
    "total_rows_used_ml",
    "threads",
    "chunksize",
    "parquet_files_size_mib",
    "mae",
    "rmse",
    "r2",
    "msg"
]

In [12]:
file = "./single-machine/results_4VCPUs_8GB/ml/csv/ml_sm.csv"
df = pd.read_csv(file)

# Keep only columns that exist in this file
numeric_columns = [c for c in numeric_cols if c in df.columns]
list_columns = [c for c in list_cols if c in df.columns]
keep_columns = [c for c in keep_cols if c in df.columns]

In [13]:
out = file.replace(".csv", "_summary.csv")

df_summary = summarize_csv_in_groups_of_three(
    df = df,
    output_csv = out,
    numeric_columns = numeric_columns,
    list_columns = list_columns,
    keep_columns = keep_columns,
    filter_column = "test_number",
    filter_threshold = 100,
)